# Week 5: Language detection, translation, England filter and reranking (Colab version)

Self-contained: choose **Runtime > Change runtime type > T4 GPU** (recommended), then **Runtime > Run all**.
It writes the project code, collects the 38 official pages live, builds the 150-question test set and reruns the week 5 experiment. It downloads three models (about 5 GB in total), so the first run takes roughly 10 minutes on a GPU.

Pages are fetched live, so numbers can differ slightly from the 2 Nov 2026 run if a page has changed. The manual translation review (section 2) refers to the 2 Nov translations.

In [ ]:
%pip install -q sentence-transformers faiss-cpu rank-bm25 lingua-language-detector sentencepiece

In [ ]:
import os, sys
for d in ['src', 'eval', 'data/raw', 'data/processed', 'results']:
    os.makedirs(d, exist_ok=True)

In [ ]:
%%writefile src/sources.py
"""Curated list of official sources for the three domains.

Every entry was checked against the live site on 28 Oct 2026 (HTTP 200).
GOV.UK pages are fetched through the public Content API; NHS pages as HTML.
Both are published under the Open Government Licence v3.0, and NHS reuse
also needs attribution, a link back and a date notice (nhs.uk terms).
"""

GOVUK = {
    "schools": [
        "schools-admissions",
        "apply-for-primary-school-place",
        "apply-for-secondary-school-place",
        "school-attendance-absence",
        "apply-free-school-meals",
        "children-with-special-educational-needs",
        "free-school-transport",
        "school-uniform",
        "types-of-school",
        "school-term-holiday-dates",
        "know-when-you-can-leave-school",
        "complain-about-school",
    ],
    "housing": [
        "private-renting",
        "private-renting-evictions",
        "guidance/renters-rights-act-overview-for-tenants",
        "tenancy-deposit-protection",
        "council-housing",
        "housing-association-homes",
        "council-housing-association-evictions",
        "housing-benefit",
        "universal-credit",
        "homelessness-help-from-council",
        "council-tax",
        "apply-council-tax-reduction",
    ],
    "health": [
        "help-nhs-costs",
        "healthcare-immigration-application",
    ],
}

NHS = {
    "health": [
        "https://www.nhs.uk/nhs-services/gps/how-to-register-with-a-gp-surgery/",
        "https://www.nhs.uk/nhs-services/gps/gp-appointments-and-bookings/",
        "https://www.nhs.uk/nhs-services/urgent-and-emergency-care-services/when-to-use-111/",
        "https://www.nhs.uk/nhs-services/urgent-and-emergency-care-services/when-to-go-to-ae/",
        "https://www.nhs.uk/nhs-services/urgent-and-emergency-care-services/when-to-call-999/",
        "https://www.nhs.uk/nhs-services/dentists/how-to-find-an-nhs-dentist/",
        "https://www.nhs.uk/nhs-services/dentists/how-much-nhs-dental-treatment-costs/",
        "https://www.nhs.uk/nhs-services/pharmacies/how-pharmacies-can-help/",
        "https://www.nhs.uk/nhs-services/help-with-health-costs/nhs-low-income-scheme-lis/",
        "https://www.nhs.uk/nhs-services/visiting-or-moving-to-england/how-to-access-nhs-services-in-england-if-you-are-visiting-from-abroad/",
        "https://www.nhs.uk/vaccinations/nhs-vaccinations-and-when-to-have-them/",
        "https://www.nhs.uk/pregnancy/finding-out/what-to-do-when-you-find-out-you-are-pregnant/",
    ],
}


In [ ]:
%%writefile src/ingest.py
"""Fetch every source page, keep a raw snapshot, and write one clean record per page.

Output: data/raw/<id>.(json|html) and data/processed/pages.jsonl
Each record keeps the section structure (h2/h3 headings) so chunks can carry
their heading, plus three dates: the publisher's own date, the publisher's
last technical update, and when we fetched it (for data-currency reporting).
"""
import hashlib
import json
import pathlib
import time
from datetime import datetime, timezone

import requests
from bs4 import BeautifulSoup

from sources import GOVUK, NHS

ROOT = pathlib.Path(__file__).resolve().parents[1]
RAW = ROOT / "data" / "raw"
OUT = ROOT / "data" / "processed" / "pages.jsonl"
UA = {"User-Agent": "MSc research prototype (multilingual civic assistant); polite crawler"}

# GOV.UK formats store text in different fields; order follows the rendered page.
GOVUK_TEXT_FIELDS = ["introduction", "introductory_paragraph", "body", "more_information",
                     "need_to_know", "other_ways_to_apply", "what_you_need_to_know"]


def page_id(url):
    return hashlib.sha1(url.encode()).hexdigest()[:10]


def sections_from_html(html, page_title):
    """Split HTML into (heading, text) sections at h2/h3 boundaries."""
    soup = BeautifulSoup(html, "html.parser")
    sections, heading, buf = [], page_title, []
    for el in soup.find_all(["h2", "h3", "p", "li", "td", "th"]):
        if el.name in ("h2", "h3"):
            if buf:
                sections.append({"heading": heading, "text": " ".join(buf)})
            heading, buf = el.get_text(" ", strip=True), []
        elif not el.find_parent(["li", "td", "th"]) or el.name in ("td", "th"):
            # skip nested elements already covered by their parent's text
            t = el.get_text(" ", strip=True)
            if t:
                buf.append(t)
    if buf:
        sections.append({"heading": heading, "text": " ".join(buf)})
    return sections


def fetch_govuk(path, domain):
    r = requests.get("https://www.gov.uk/api/content/" + path, headers=UA, timeout=30)
    r.raise_for_status()
    data = r.json()
    url = "https://www.gov.uk/" + path
    (RAW / f"{page_id(url)}.json").write_text(r.text)
    det = data.get("details", {})
    html = "".join(det.get(f, "") for f in GOVUK_TEXT_FIELDS if isinstance(det.get(f), str))
    for part in det.get("parts", []):
        html += f"<h2>{part['title']}</h2>{part['body']}"
    title = data.get("title", "")
    return {
        "id": page_id(url), "url": url, "source": "GOV.UK", "domain": domain,
        "title": title, "format": data.get("schema_name"),
        "public_updated_at": (data.get("public_updated_at") or "")[:10],
        "updated_at": (data.get("updated_at") or "")[:10],
        "sections": sections_from_html(html, title),
    }


def fetch_nhs(url, domain):
    r = requests.get(url, headers=UA, timeout=30)
    r.raise_for_status()
    (RAW / f"{page_id(url)}.html").write_text(r.text)
    soup = BeautifulSoup(r.text, "html.parser")
    reviewed = soup.find(string=lambda s: s and "Page last reviewed" in s)
    reviewed = reviewed.strip().split(":", 1)[1].strip() if reviewed else ""
    modified = ""
    for s in soup.find_all("script", type="application/ld+json"):
        try:
            modified = (json.loads(s.string or "{}").get("dateModified") or modified)[:10]
        except (ValueError, AttributeError):
            pass
    article = soup.find("article") or soup.find("main")
    for junk in article.find_all(["aside", "nav", "footer", "script", "style"]):
        junk.decompose()
    title = soup.find("h1").get_text(" ", strip=True)
    # the lead sentence ("999 is for life-threatening emergencies...") sits outside <article>
    lede = soup.find(class_="nhsuk-lede-text")
    html = (str(lede) if lede and article not in lede.parents else "") + str(article)
    return {
        "id": page_id(url), "url": url, "source": "NHS.uk", "domain": domain,
        "title": title, "format": "nhs_article",
        "public_updated_at": datetime.strptime(reviewed, "%d %B %Y").strftime("%Y-%m-%d") if reviewed else "",
        "updated_at": modified,
        "sections": sections_from_html(html, title),
    }


def main():
    RAW.mkdir(parents=True, exist_ok=True)
    OUT.parent.mkdir(parents=True, exist_ok=True)
    fetched_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
    jobs = [(fetch_govuk, p, d) for d, ps in GOVUK.items() for p in ps]
    jobs += [(fetch_nhs, u, d) for d, us in NHS.items() for u in us]
    records, failed = [], []
    for fn, target, domain in jobs:
        try:
            rec = fn(target, domain)
            rec["fetched_at"] = fetched_at
            rec["words"] = sum(len(s["text"].split()) for s in rec["sections"])
            records.append(rec)
            print(f"ok   {rec['domain']:8} {rec['words']:5}w  {rec['title']}")
        except Exception as e:  # keep going; report at the end
            failed.append((target, repr(e)))
            print(f"FAIL {target}: {e}")
        time.sleep(0.5)
    with OUT.open("w") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"\n{len(records)} pages written to {OUT.relative_to(ROOT)}; {len(failed)} failed")
    for t, e in failed:
        print("  failed:", t, e)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/chunk.py
"""Turn pages into retrievable chunks that respect the page's own sections.

Rules (kept simple so they are easy to describe in the methodology):
  * one chunk per h2/h3 section;
  * each section is tagged with the UK nation it applies to (from its heading);
  * sections shorter than MIN_WORDS are merged into the next one on the same page,
    but only when both have the same nation tag;
  * sections longer than MAX_WORDS are split into windows with OVERLAP words;
  * every chunk text starts with "<page title> - <section heading>:" so the
    embedding sees the context a reader would see.
Output: data/processed/chunks.csv
"""
import json
import pathlib
import re

import pandas as pd

ROOT = pathlib.Path(__file__).resolve().parents[1]
MIN_WORDS, MAX_WORDS, OVERLAP = 40, 200, 40
OTHER_NATIONS = ["Scotland", "Wales", "Northern Ireland"]


def nation_of(heading):
    """GOV.UK marks nation-specific sections in the heading ("Scotland", "If you're in Wales...").
    Everything else applies to England (or the whole UK). NHS.uk pages cover England only."""
    named = [n for n in OTHER_NATIONS if re.search(rf"\b{n}\b", heading)]
    return ";".join(n.lower().replace(" ", "_") for n in named) or "england"


def join(a, b):
    return {"heading": a["heading"] + " / " + b["heading"], "text": a["text"] + " " + b["text"],
            "nation": a["nation"]}


def merge_small(sections):
    merged, carry = [], None
    for s in sections:
        s = {**s, "nation": nation_of(s["heading"])}
        if carry and carry["nation"] == s["nation"]:
            s = join(carry, s)
        elif carry:
            merged.append(carry)  # never merge across nations; keep the short one alone
        carry = None
        if len(s["text"].split()) < MIN_WORDS:
            carry = s
        else:
            merged.append(s)
    if carry:  # trailing short section joins the previous one if same nation, or stands alone
        if merged and merged[-1]["nation"] == carry["nation"]:
            merged[-1] = join(merged[-1], carry)
        else:
            merged.append(carry)
    return merged


def windows(words):
    if len(words) <= MAX_WORDS:
        return [words]
    step = MAX_WORDS - OVERLAP
    return [words[i:i + MAX_WORDS] for i in range(0, len(words) - OVERLAP, step)]


def main():
    pages = [json.loads(l) for l in open(ROOT / "data/processed/pages.jsonl")]
    rows = []
    for p in pages:
        for s_i, s in enumerate(merge_small(p["sections"])):
            for w_i, w in enumerate(windows(s["text"].split())):
                body = " ".join(w)
                rows.append({
                    "chunk_id": f"{p['id']}-{s_i:02d}-{w_i}",
                    "page_id": p["id"], "url": p["url"], "source": p["source"], "domain": p["domain"],
                    "title": p["title"], "heading": s["heading"], "nation": s["nation"],
                    "public_updated_at": p["public_updated_at"], "fetched_at": p["fetched_at"],
                    "words": len(w), "body": body,
                    "text": f"{p['title']} - {s['heading']}: {body}",
                })
    df = pd.DataFrame(rows)
    df.to_csv(ROOT / "data/processed/chunks.csv", index=False)
    print(f"{len(pages)} pages -> {len(df)} chunks")
    print(df.groupby("domain").agg(pages=("page_id", "nunique"), chunks=("chunk_id", "count"),
                                   words=("words", "sum")).to_string())
    print("nation tags:", df.nation.value_counts().to_dict())
    print("chunk words: median", int(df.words.median()), "min", df.words.min(), "max", df.words.max())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/retrieval.py
"""Dense and lexical retrievers over the chunk table.

Dense: sentence-transformers model + FAISS inner-product index on normalised
vectors (= cosine similarity). E5 models need "query: " / "passage: " prefixes.
Lexical: BM25 (rank_bm25), used as a non-neural baseline.
"""
import re

import torch  # noqa: F401  must load before faiss: on macOS the two OpenMP runtimes otherwise crash NLLB generation
import faiss
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer


class DenseRetriever:
    def __init__(self, model_name, texts):
        self.model = SentenceTransformer(model_name)
        self.e5 = "e5" in model_name.lower()
        passages = [("passage: " + t) if self.e5 else t for t in texts]
        emb = self.model.encode(passages, normalize_embeddings=True, batch_size=32, show_progress_bar=False)
        self.index = faiss.IndexFlatIP(emb.shape[1])
        self.index.add(np.asarray(emb, dtype="float32"))

    def search(self, query, k=10):
        q = self.model.encode([("query: " + query) if self.e5 else query], normalize_embeddings=True)
        scores, ids = self.index.search(np.asarray(q, dtype="float32"), k)
        return list(zip(ids[0].tolist(), scores[0].tolist()))

    def save(self, path):
        faiss.write_index(self.index, str(path))


def tokenize(text):
    return re.findall(r"\w+", text.lower())


class BM25Retriever:
    def __init__(self, texts):
        self.bm25 = BM25Okapi([tokenize(t) for t in texts])

    def search(self, query, k=10):
        scores = self.bm25.get_scores(tokenize(query))
        ids = np.argsort(-scores)[:k]
        return [(int(i), float(scores[i])) for i in ids]


In [ ]:
%%writefile src/language.py
"""Detect which supported language a question is written in.

Uses lingua (Apache-2.0), restricted to the five project languages, which makes
short questions much easier to classify than open-set detection. Arabic and Urdu
share a script, so restricting the set matters most for those two.
"""
from lingua import Language, LanguageDetectorBuilder

SUPPORTED = {
    Language.ENGLISH: "en", Language.POLISH: "pl", Language.ROMANIAN: "ro",
    Language.ARABIC: "ar", Language.URDU: "ur",
}
_detector = LanguageDetectorBuilder.from_languages(*SUPPORTED).build()


def detect(text):
    """Return 'en', 'pl', 'ro', 'ar' or 'ur'; falls back to 'en' if lingua cannot decide."""
    lang = _detector.detect_language_of(text)
    return SUPPORTED.get(lang, "en")


In [ ]:
%%writefile src/translate.py
"""Machine translation with NLLB-200 (facebook/nllb-200-distilled-600M, CC-BY-NC-4.0).

Used in week 5 to translate questions into English before retrieval, and in week 6
for answers going back into the user's language. Non-commercial licence: fine for
this research prototype, not for a production service.
"""
import re

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL = "facebook/nllb-200-distilled-600M"
NLLB_CODES = {"en": "eng_Latn", "pl": "pol_Latn", "ro": "ron_Latn", "ar": "arb_Arab", "ur": "urd_Arab"}


class Translator:
    def __init__(self, model_name=MODEL):
        self.tok = AutoTokenizer.from_pretrained(model_name)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"  # GPU on Colab, CPU elsewhere
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name).eval().to(self.device)

    @torch.inference_mode()
    def _translate_sentences(self, sentences, src, tgt, max_length):
        self.tok.src_lang = NLLB_CODES[src]
        batch = self.tok(sentences, return_tensors="pt", padding=True, truncation=True, max_length=400).to(self.device)
        out = self.model.generate(
            **batch, max_length=max_length, num_beams=4,
            forced_bos_token_id=self.tok.convert_tokens_to_ids(NLLB_CODES[tgt]))
        return self.tok.batch_decode(out, skip_special_tokens=True)

    def translate(self, texts, src, tgt="en", max_length=256):
        """Translate a list of strings from language `src` to `tgt` (project codes, e.g. 'pl').

        NLLB is trained on single sentences and silently drops later sentences of a
        multi-sentence input, so each text is split into sentences first and rejoined.
        """
        if src == tgt:
            return list(texts)
        split = [split_sentences(t) for t in texts]
        flat = [s for parts in split for s in parts]
        done = iter(self._translate_sentences(flat, src, tgt, max_length))
        return [" ".join(next(done) for _ in parts) for parts in split]


def split_sentences(text):
    """Split on sentence-final punctuation, including Arabic ؟ and Urdu ۔ ."""
    parts = [p.strip() for p in re.split(r"(?<=[.!?؟۔])\s+", text.strip())]
    return [p for p in parts if p] or [text]


In [ ]:
%%writefile src/rerank.py
"""Second-stage reranking with a multilingual cross-encoder (BAAI/bge-reranker-v2-m3, Apache-2.0).

The bi-encoder (e5) compares a question vector with pre-computed chunk vectors, which
is fast but coarse. A cross-encoder reads the question and one chunk together and
scores how well that chunk answers it; it is too slow for the whole corpus, so it
only reorders the top candidates from the first stage.
"""
from sentence_transformers import CrossEncoder

MODEL = "BAAI/bge-reranker-v2-m3"


class Reranker:
    def __init__(self, model_name=MODEL):
        self.model = CrossEncoder(model_name, max_length=512)

    def rerank(self, query, candidates, texts):
        """candidates: list of chunk ids; texts: chunk id -> text. Returns [(id, score)] best first."""
        if not candidates:
            return []
        scores = self.model.predict([(query, texts[i]) for i in candidates], batch_size=16, show_progress_bar=False)
        return sorted(zip(candidates, map(float, scores)), key=lambda x: -x[1])


In [ ]:
%%writefile src/pipeline.py
"""Week 5 search pipeline: question in any supported language -> best official passages.

    question
      -> detect language                       (language.py)
      -> translate to English if not English   (translate.py, optional)
      -> dense retrieval, original + English   (retrieval.py, e5 + FAISS)
      -> fuse the two ranked lists (RRF)
      -> keep England / UK-wide chunks only     (nation tag from chunk.py)
      -> rerank top candidates                  (rerank.py, optional)
      -> top k passages with title, heading, URL and dates for citation
"""
import pathlib

import pandas as pd

from language import detect
from retrieval import DenseRetriever

ROOT = pathlib.Path(__file__).resolve().parents[1]
E5 = "intfloat/multilingual-e5-base"
RRF_K = 60  # standard constant from Cormack et al. (2009)


def rrf(*ranked_lists, k=RRF_K):
    """Reciprocal rank fusion: combine ranked lists of ids into one ranking."""
    score = {}
    for ranked in ranked_lists:
        for rank, i in enumerate(ranked, start=1):
            score[i] = score.get(i, 0.0) + 1.0 / (k + rank)
    return sorted(score, key=lambda i: -score[i])


class SearchPipeline:
    def __init__(self, chunks=None, translate=True, rerank=True, england_only=True, pool=20):
        self.chunks = chunks if chunks is not None else pd.read_csv(ROOT / "data/processed/chunks.csv")
        self.texts = self.chunks.text.tolist()
        self.retriever = DenseRetriever(E5, self.texts)
        self.allowed = set(self.chunks.index[self.chunks.nation == "england"]) if england_only else None
        self.pool = pool
        self.translator = None
        if translate:
            from translate import Translator
            self.translator = Translator()
        self.reranker = None
        if rerank:
            from rerank import Reranker
            self.reranker = Reranker()

    def _dense(self, query, n):
        return [i for i, _ in self.retriever.search(query, k=n)]

    def search(self, question, k=3, lang=None):
        lang = lang or detect(question)
        english = question
        lists = [self._dense(question, 50)]
        if self.translator and lang != "en":
            english = self.translator.translate([question], src=lang)[0]
            lists.append(self._dense(english, 50))
        ranked = rrf(*lists) if len(lists) > 1 else lists[0]
        if self.allowed is not None:
            ranked = [i for i in ranked if i in self.allowed]
        ranked = ranked[:self.pool]
        if self.reranker:
            ranked = [i for i, _ in self.reranker.rerank(question, ranked, self.texts)]
        results = []
        for i in ranked[:k]:
            c = self.chunks.iloc[i]
            results.append({"chunk": int(i), "title": c.title, "heading": c.heading, "url": c.url,
                            "source": c.source, "nation": c.nation, "public_updated_at": c.public_updated_at,
                            "fetched_at": c.fetched_at, "text": c.body})
        return {"lang": lang, "english_query": english, "results": results}


In [ ]:
%%writefile eval/build_queries.py
"""Build the retrieval test set and check every answer phrase against the corpus.

Each English question has:
  gold_url - the page that answers it
  evidence - a short phrase copied verbatim from that page that contains the answer
A retrieved chunk only counts as "answer-bearing" if it contains the evidence
phrase, so a hit on the right page but the wrong section is not counted.

Translations (pl, ro, ar, ur) were written by the author and are flagged
"unverified" until a native speaker checks them.
Output: eval/queries.csv
"""
import csv
import json
import pathlib
import sys

ROOT = pathlib.Path(__file__).resolve().parents[1]
G = "https://www.gov.uk/"
N = "https://www.nhs.uk/nhs-services/"

# qid: (domain, gold_url, evidence, {lang: question})
Q = {
    "S01": ("schools", G + "schools-admissions", "close on 15 January", {
        "en": "When is the deadline to apply for a primary school place?",
        "pl": "Jaki jest termin składania wniosku o miejsce w szkole podstawowej?",
        "ro": "Care este termenul limită pentru a depune cererea pentru un loc la școala primară?",
        "ar": "ما هو الموعد النهائي للتقديم على مقعد في المدرسة الابتدائية؟",
        "ur": "پرائمری اسکول میں داخلے کی درخواست دینے کی آخری تاریخ کیا ہے؟"}),
    "S02": ("schools", G + "schools-admissions", "by 31 October", {
        "en": "What is the last date to apply for a secondary school place?",
        "pl": "Do kiedy trzeba złożyć wniosek o miejsce w szkole średniej?",
        "ro": "Până la ce dată trebuie să aplic pentru un loc la școala secundară?",
        "ar": "ما هو آخر موعد للتقديم على مقعد في المدرسة الثانوية؟",
        "ur": "سیکنڈری اسکول میں داخلے کے لیے درخواست کس تاریخ تک دینی ہوگی؟"}),
    "S03": ("schools", G + "schools-admissions", "primary schools on 16 April", {
        "en": "When will I find out which primary school my child has been offered?",
        "pl": "Kiedy dowiem się, do której szkoły podstawowej zostało przyjęte moje dziecko?",
        "ro": "Când voi afla la ce școală primară a primit loc copilul meu?",
        "ar": "متى سأعرف المدرسة الابتدائية التي قُبل فيها طفلي؟",
        "ur": "مجھے کب پتہ چلے گا کہ میرے بچے کو کون سے پرائمری اسکول میں جگہ ملی ہے؟"}),
    "S04": ("schools", G + "schools-admissions", "at least 20 school days", {
        "en": "How long do I have to appeal if my child is refused a school place?",
        "pl": "Ile mam czasu na odwołanie, jeśli moje dziecko nie dostało miejsca w szkole?",
        "ro": "Cât timp am la dispoziție să fac contestație dacă copilul meu nu a primit loc la școală?",
        "ar": "كم لدي من الوقت للاستئناف إذا رُفض طلب طفلي للحصول على مقعد في المدرسة؟",
        "ur": "اگر میرے بچے کو اسکول میں جگہ نہ ملے تو اپیل کرنے کے لیے میرے پاس کتنا وقت ہے؟"}),
    "S05": ("schools", G + "school-attendance-absence", "fined or face legal action for taking your child on holiday", {
        "en": "Can I be fined for taking my child on holiday during term time?",
        "pl": "Czy mogę dostać grzywnę za zabranie dziecka na wakacje w trakcie roku szkolnego?",
        "ro": "Pot primi amendă dacă îmi iau copilul în vacanță în timpul semestrului?",
        "ar": "هل يمكن أن أُغرَّم إذا أخذت طفلي في عطلة أثناء الفصل الدراسي؟",
        "ur": "کیا ٹرم کے دوران بچے کو چھٹیوں پر لے جانے پر مجھے جرمانہ ہو سکتا ہے؟"}),
    "S06": ("schools", G + "apply-free-school-meals", "free school meals if you get any of the following", {
        "en": "Can my child get free school meals if I receive Universal Credit?",
        "pl": "Czy moje dziecko może dostać darmowe posiłki w szkole, jeśli otrzymuję Universal Credit?",
        "ro": "Poate copilul meu să primească mese gratuite la școală dacă primesc Universal Credit?",
        "ar": "هل يمكن لطفلي الحصول على وجبات مدرسية مجانية إذا كنت أتلقى Universal Credit؟",
        "ur": "اگر مجھے Universal Credit ملتا ہے تو کیا میرے بچے کو اسکول میں مفت کھانا مل سکتا ہے؟"}),
    "S07": ("schools", G + "free-school-transport", "more than 2 miles away and the child is under 8", {
        "en": "Is school transport free if the school is far from home?",
        "pl": "Czy transport do szkoły jest darmowy, jeśli szkoła jest daleko od domu?",
        "ro": "Este transportul la școală gratuit dacă școala este departe de casă?",
        "ar": "هل المواصلات المدرسية مجانية إذا كانت المدرسة بعيدة عن المنزل؟",
        "ur": "اگر اسکول گھر سے دور ہو تو کیا اسکول کی ٹرانسپورٹ مفت ہے؟"}),
    "S08": ("schools", G + "know-when-you-can-leave-school", "16 by the end of the summer holidays", {
        "en": "At what age can my child leave school?",
        "pl": "W jakim wieku moje dziecko może skończyć szkołę?",
        "ro": "La ce vârstă poate copilul meu să termine școala?",
        "ar": "في أي عمر يمكن لطفلي ترك المدرسة؟",
        "ur": "میرا بچہ کس عمر میں اسکول چھوڑ سکتا ہے؟"}),
    "S09": ("schools", G + "children-with-special-educational-needs", "SEN support - support given in school", {
        "en": "What support can my child get for special educational needs?",
        "pl": "Jakie wsparcie może otrzymać moje dziecko ze specjalnymi potrzebami edukacyjnymi?",
        "ro": "Ce sprijin poate primi copilul meu pentru cerințe educaționale speciale?",
        "ar": "ما الدعم الذي يمكن أن يحصل عليه طفلي إذا كانت لديه احتياجات تعليمية خاصة؟",
        "ur": "خصوصی تعلیمی ضروریات والے میرے بچے کو کیا مدد مل سکتی ہے؟"}),
    "S10": ("schools", G + "complain-about-school", "different ways to complain in England", {
        "en": "How do I complain about my child's state school?",
        "pl": "Jak złożyć skargę na państwową szkołę mojego dziecka?",
        "ro": "Cum fac o plângere împotriva școlii de stat a copilului meu?",
        "ar": "كيف أقدم شكوى ضد المدرسة الحكومية التي يدرس فيها طفلي؟",
        "ur": "میں اپنے بچے کے سرکاری اسکول کے بارے میں شکایت کیسے کروں؟"}),

    "H01": ("housing", G + "guidance/renters-rights-act-overview-for-tenants", "section 21 process", {
        "en": "Can my landlord evict me without giving a reason?",
        "pl": "Czy wynajmujący może mnie eksmitować bez podania powodu?",
        "ro": "Poate proprietarul să mă evacueze fără să-mi dea un motiv?",
        "ar": "هل يمكن للمالك أن يطردني دون إبداء سبب؟",
        "ur": "کیا مالک مکان مجھے بغیر کوئی وجہ بتائے گھر سے نکال سکتا ہے؟"}),
    "H02": ("housing", G + "guidance/renters-rights-act-overview-for-tenants", "increase the rent once a year", {
        "en": "How often can my landlord increase the rent?",
        "pl": "Jak często wynajmujący może podnosić czynsz?",
        "ro": "Cât de des poate proprietarul să crească chiria?",
        "ar": "كم مرة يمكن للمالك أن يرفع الإيجار؟",
        "ur": "مالک مکان کتنی بار کرایہ بڑھا سکتا ہے؟"}),
    "H03": ("housing", G + "tenancy-deposit-protection", "Deposit Protection Service", {
        "en": "My landlord took a deposit from me. Does it have to be protected?",
        "pl": "Wynajmujący pobrał ode mnie kaucję. Czy musi ona być zabezpieczona?",
        "ro": "Proprietarul mi-a luat o garanție. Trebuie aceasta să fie protejată?",
        "ar": "أخذ المالك مني مبلغ تأمين، هل يجب أن يكون هذا التأمين محميًا؟",
        "ur": "مالک مکان نے مجھ سے ڈپازٹ لیا ہے، کیا اس ڈپازٹ کا محفوظ ہونا ضروری ہے؟"}),
    "H04": ("housing", G + "housing-benefit", "help you pay your rent", {
        "en": "How can I get help paying my rent if I am on a low income?",
        "pl": "Jak uzyskać pomoc w opłaceniu czynszu, jeśli mam niskie dochody?",
        "ro": "Cum pot primi ajutor pentru plata chiriei dacă am un venit mic?",
        "ar": "كيف أحصل على مساعدة في دفع الإيجار إذا كان دخلي منخفضًا؟",
        "ur": "کم آمدنی کی صورت میں مجھے کرایہ ادا کرنے میں مدد کیسے مل سکتی ہے؟"}),
    "H05": ("housing", G + "council-housing", "join a waiting list", {
        "en": "How do I apply for council housing?",
        "pl": "Jak złożyć wniosek o mieszkanie komunalne?",
        "ro": "Cum aplic pentru o locuință socială de la consiliul local?",
        "ar": "كيف أتقدم بطلب للحصول على سكن من المجلس المحلي؟",
        "ur": "میں کونسل کے مکان کے لیے درخواست کیسے دوں؟"}),
    "H06": ("housing", G + "homelessness-help-from-council", "homeless or about to lose your home", {
        "en": "I am about to lose my home. Can the council help me?",
        "pl": "Zaraz stracę dom. Czy lokalny urząd może mi pomóc?",
        "ro": "Sunt pe punctul de a-mi pierde locuința. Mă poate ajuta consiliul local?",
        "ar": "أنا على وشك أن أفقد منزلي، هل يمكن للمجلس المحلي مساعدتي؟",
        "ur": "میرا گھر چھننے والا ہے، کیا کونسل میری مدد کر سکتی ہے؟"}),
    "H07": ("housing", G + "private-renting", "safe and in a good state of repair", {
        "en": "Does my landlord have to keep my rented home in good repair?",
        "pl": "Czy wynajmujący musi utrzymywać wynajmowane mieszkanie w dobrym stanie technicznym?",
        "ro": "Este proprietarul obligat să mențină locuința închiriată în stare bună?",
        "ar": "هل يجب على المالك أن يحافظ على المنزل المؤجر في حالة جيدة؟",
        "ur": "کیا مالک مکان پر کرائے کے گھر کو اچھی حالت میں رکھنا لازم ہے؟"}),
    "H08": ("housing", G + "private-renting", "24 hours", {
        "en": "How much notice must my landlord give before visiting the property?",
        "pl": "Z jakim wyprzedzeniem wynajmujący musi mnie uprzedzić przed wizytą w mieszkaniu?",
        "ro": "Cu cât timp înainte trebuie să mă anunțe proprietarul dacă vrea să vină în locuință?",
        "ar": "كم من الوقت يجب أن يخطرني المالك قبل أن يزور المنزل؟",
        "ur": "گھر آنے سے پہلے مالک مکان کو مجھے کتنا پہلے اطلاع دینی چاہیے؟"}),
    "H09": ("housing", G + "council-tax", "25% off your bill", {
        "en": "Do I get a council tax discount if I live alone?",
        "pl": "Czy dostanę zniżkę na podatek lokalny, jeśli mieszkam sam?",
        "ro": "Primesc reducere la impozitul local pe locuință dacă locuiesc singur?",
        "ar": "هل أحصل على خصم في ضريبة المجلس المحلي إذا كنت أعيش وحدي؟",
        "ur": "اگر میں اکیلا رہتا ہوں تو کیا مجھے کونسل ٹیکس میں رعایت ملتی ہے؟"}),
    "H10": ("housing", G + "council-housing-association-evictions", "stop or delay the eviction", {
        "en": "What can I do if my council or housing association wants to evict me?",
        "pl": "Co mogę zrobić, jeśli lokalny urząd lub stowarzyszenie mieszkaniowe chce mnie eksmitować?",
        "ro": "Ce pot face dacă consiliul local sau asociația de locuințe vrea să mă evacueze?",
        "ar": "ماذا يمكنني أن أفعل إذا أراد المجلس المحلي أو جمعية الإسكان طردي؟",
        "ur": "اگر کونسل یا ہاؤسنگ ایسوسی ایشن مجھے گھر سے نکالنا چاہے تو میں کیا کر سکتا ہوں؟"}),

    "M01": ("health", N + "gps/how-to-register-with-a-gp-surgery/", "change your current GP surgery online", {
        "en": "How do I register with a GP surgery?",
        "pl": "Jak zarejestrować się u lekarza rodzinnego?",
        "ro": "Cum mă înregistrez la un medic de familie?",
        "ar": "كيف أسجل لدى طبيب عام؟",
        "ur": "میں جی پی کے پاس رجسٹریشن کیسے کرواؤں؟"}),
    "M02": ("health", N + "gps/how-to-register-with-a-gp-surgery/", "do not need ID, proof of address", {
        "en": "Do I need proof of address or immigration status to register with a GP?",
        "pl": "Czy potrzebuję potwierdzenia adresu lub statusu imigracyjnego, aby zarejestrować się u lekarza rodzinnego?",
        "ro": "Am nevoie de dovada adresei sau a statutului de imigrant ca să mă înregistrez la medicul de familie?",
        "ar": "هل أحتاج إلى إثبات عنوان أو إثبات وضع الهجرة للتسجيل لدى طبيب عام؟",
        "ur": "کیا جی پی کے پاس رجسٹر ہونے کے لیے مجھے پتے یا امیگریشن اسٹیٹس کا ثبوت چاہیے؟"}),
    "M03": ("health", N + "urgent-and-emergency-care-services/when-to-use-111/", "if you think you need medical help right now", {
        "en": "When should I use NHS 111?",
        "pl": "Kiedy powinienem skorzystać z NHS 111?",
        "ro": "Când ar trebui să folosesc NHS 111?",
        "ar": "متى يجب أن أستخدم خدمة NHS 111؟",
        "ur": "مجھے NHS 111 کب استعمال کرنا چاہیے؟"}),
    "M04": ("health", N + "urgent-and-emergency-care-services/when-to-go-to-ae/", "serious injuries and life-threatening emergencies", {
        "en": "When should I go to A&E?",
        "pl": "Kiedy powinienem iść na szpitalny oddział ratunkowy?",
        "ro": "Când ar trebui să merg la urgențe?",
        "ar": "متى يجب أن أذهب إلى قسم الطوارئ؟",
        "ur": "مجھے ہسپتال کے ایمرجنسی شعبے میں کب جانا چاہیے؟"}),
    "M05": ("health", N + "urgent-and-emergency-care-services/when-to-call-999/", "999 is for life-threatening emergencies", {
        "en": "When should I call 999?",
        "pl": "Kiedy powinienem dzwonić pod numer 999?",
        "ro": "Când ar trebui să sun la 999?",
        "ar": "متى يجب أن أتصل بالرقم 999؟",
        "ur": "مجھے 999 پر کب کال کرنی چاہیے؟"}),
    "M06": ("health", N + "dentists/how-to-find-an-nhs-dentist/", "Ask for an NHS appointment", {
        "en": "How can I find an NHS dentist?",
        "pl": "Jak znaleźć dentystę NHS?",
        "ro": "Cum găsesc un dentist NHS?",
        "ar": "كيف أجد طبيب أسنان تابعًا لـ NHS؟",
        "ur": "میں NHS کا ڈینٹسٹ کیسے تلاش کروں؟"}),
    "M07": ("health", N + "dentists/how-much-nhs-dental-treatment-costs/", "£27.90", {
        "en": "How much does a basic NHS dental check-up cost?",
        "pl": "Ile kosztuje podstawowy przegląd u dentysty w NHS?",
        "ro": "Cât costă un control stomatologic de bază la NHS?",
        "ar": "كم يكلف فحص الأسنان الأساسي في NHS؟",
        "ur": "NHS میں دانتوں کے بنیادی معائنے پر کتنا خرچ آتا ہے؟"}),
    "M08": ("health", N + "pharmacies/how-pharmacies-can-help/", "This is called Pharmacy First", {
        "en": "Can a pharmacy give me prescription medicine without seeing a GP?",
        "pl": "Czy apteka może wydać mi lek na receptę bez wizyty u lekarza?",
        "ro": "Poate o farmacie să-mi dea medicamente cu rețetă fără să merg la medic?",
        "ar": "هل يمكن للصيدلية أن تعطيني دواءً يحتاج إلى وصفة طبية دون أن أرى الطبيب؟",
        "ur": "کیا فارمیسی ڈاکٹر کو دکھائے بغیر مجھے نسخے والی دوا دے سکتی ہے؟"}),
    "M09": ("health", N + "help-with-health-costs/nhs-low-income-scheme-lis/", "HC2 certificate", {
        "en": "Can I get help with health costs if I have a low income?",
        "pl": "Czy mogę dostać pomoc w pokryciu kosztów leczenia, jeśli mam niskie dochody?",
        "ro": "Pot primi ajutor pentru costurile medicale dacă am venituri mici?",
        "ar": "هل يمكنني الحصول على مساعدة في التكاليف الصحية إذا كان دخلي منخفضًا؟",
        "ur": "کیا کم آمدنی کی صورت میں مجھے علاج کے اخراجات میں مدد مل سکتی ہے؟"}),
    "M10": ("health", G + "healthcare-immigration-application", "£776 per year for students", {
        "en": "How much is the immigration health surcharge for students?",
        "pl": "Ile wynosi dopłata zdrowotna dla imigrantów w przypadku studentów?",
        "ro": "Cât costă taxa de sănătate pentru imigranți în cazul studenților?",
        "ar": "كم تبلغ رسوم الرعاية الصحية للمهاجرين بالنسبة للطلاب؟",
        "ur": "طلبہ کے لیے امیگریشن ہیلتھ سرچارج کتنا ہے؟"}),
}


def main():
    pages = {}
    for line in open(ROOT / "data/processed/pages.jsonl"):
        r = json.loads(line)
        pages[r["url"]] = " ".join(s["text"] for s in r["sections"]).lower()
    bad = [(q, url) for q, (_, url, ev, _) in Q.items() if url not in pages or ev.lower() not in pages[url]]
    if bad:
        sys.exit(f"evidence not found on gold page: {bad}")
    out = ROOT / "eval/queries.csv"
    with out.open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["qid", "domain", "lang", "question", "gold_url", "evidence", "translation_status"])
        for qid, (domain, url, ev, qs) in Q.items():
            for lang, text in qs.items():
                w.writerow([qid, domain, lang, text, url, ev, "source" if lang == "en" else "unverified"])
    print(f"{len(Q)} questions x {len(next(iter(Q.values()))[3])} languages -> {out.relative_to(ROOT)}; "
          f"all evidence phrases found verbatim on their gold page")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile eval/evaluate_week5.py
"""Week 5 experiment: language ID, translation and reranking on the 150-query test set.

Steps
  1. Language identification accuracy (lingua, 5 languages).
  2. Translate the 120 non-English questions to English with NLLB-200 (cached to
     eval/translations_nllb.csv so a native speaker can review them too).
  3. Compare retrieval set-ups on the same nation-tagged section chunks:
       A direct                 e5 on the original question (week 4 best set-up)
       B direct+filter          A, England / UK-wide chunks only
       C translated+filter      e5 on the NLLB English translation
       D fusion+filter          reciprocal rank fusion of A and C
       E direct+filter+rerank   B, top 20 reordered by bge-reranker-v2-m3
       F fusion+filter+rerank   D, top 20 reordered by bge-reranker-v2-m3
Metrics as week 4: hit@1, hit@3, mrr@10 (gold page) and ans@3 (verbatim answer phrase in a top-3 chunk).
Outputs: results/week5_*.csv
"""
import pathlib
import sys
import time

import pandas as pd

ROOT = pathlib.Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from language import detect  # noqa: E402
from pipeline import E5, rrf  # noqa: E402
from rerank import Reranker  # noqa: E402
from retrieval import DenseRetriever  # noqa: E402
from translate import Translator  # noqa: E402

POOL = 20
OUT = ROOT / "results"


def metrics(ranked, chunks, q):
    urls = list(dict.fromkeys(chunks.url.iloc[i] for i in ranked))[:10]
    rank = urls.index(q.gold_url) + 1 if q.gold_url in urls else None
    top3 = [chunks.body.iloc[i].lower() for i in ranked[:3]]
    return {"hit@1": rank == 1, "hit@3": rank is not None and rank <= 3, "mrr@10": 1 / rank if rank else 0.0,
            "ans@3": any(q.evidence.lower() in t for t in top3),
            "top1_nation": chunks.nation.iloc[ranked[0]], "top1_url": urls[0]}


def main():
    queries = pd.read_csv(ROOT / "eval/queries.csv")
    chunks = pd.read_csv(ROOT / "data/processed/chunks.csv")
    texts = chunks.text.tolist()
    england = set(chunks.index[chunks.nation == "england"])
    OUT.mkdir(exist_ok=True)

    # 1. language identification
    queries["detected"] = [detect(q) for q in queries.question]
    lid = (queries.detected == queries.lang).groupby(queries.lang).mean().round(3)
    print("Language ID accuracy:", lid.to_dict(), "| overall", round((queries.detected == queries.lang).mean(), 3))
    lid.rename("accuracy").to_csv(OUT / "week5_language_id.csv")

    # 2. translation (cached)
    cache = ROOT / "eval/translations_nllb.csv"
    if cache.exists():
        tr = pd.read_csv(cache)
    else:
        translator, rows, t0 = Translator(), [], time.time()
        for lang, grp in queries[queries.lang != "en"].groupby("lang"):
            outs = translator.translate(grp.question.tolist(), src=lang)
            rows += [{"qid": r.qid, "lang": lang, "question": r.question, "nllb_english": o}
                     for r, o in zip(grp.itertuples(), outs)]
        tr = pd.DataFrame(rows)
        en = queries[queries.lang == "en"].set_index("qid").question
        tr["original_english"] = tr.qid.map(en)
        tr.to_csv(cache, index=False)
        print(f"translated {len(tr)} questions in {time.time() - t0:.0f}s")
    english = {(r.qid, r.lang): r.nllb_english for r in tr.itertuples()}

    # 3. retrieval set-ups
    dense = DenseRetriever(E5, texts)
    reranker = Reranker()
    rows, timing = [], {"rerank": []}
    for q in queries.itertuples():
        direct = [i for i, _ in dense.search(q.question, k=50)]
        eng_q = english.get((q.qid, q.lang), q.question)
        translated = [i for i, _ in dense.search(eng_q, k=50)] if q.lang != "en" else direct
        fused = rrf(direct, translated) if q.lang != "en" else direct
        keep = lambda r: [i for i in r if i in england]  # noqa: E731
        t0 = time.time()
        rr_direct = [i for i, _ in reranker.rerank(q.question, keep(direct)[:POOL], texts)]
        rr_fused = [i for i, _ in reranker.rerank(q.question, keep(fused)[:POOL], texts)]
        timing["rerank"].append((time.time() - t0) / 2)
        setups = {
            "A direct": direct, "B direct+filter": keep(direct), "C translated+filter": keep(translated),
            "D fusion+filter": keep(fused), "E direct+filter+rerank": rr_direct,
            "F fusion+filter+rerank": rr_fused,
        }
        for name, ranked in setups.items():
            rows.append({"system": name, "qid": q.qid, "domain": q.domain, "lang": q.lang, **metrics(ranked, chunks, q)})

    per_q = pd.DataFrame(rows)
    per_q.to_csv(OUT / "week5_per_query.csv", index=False)
    m = ["hit@1", "hit@3", "mrr@10", "ans@3"]
    overall = per_q.groupby("system")[m].mean().round(3)
    by_lang = per_q.groupby(["system", "lang"])[m].mean().round(3).reset_index()
    pd.concat([by_lang, overall.assign(lang="ALL").reset_index()]).to_csv(OUT / "week5_by_lang.csv", index=False)

    print(f"\nmean rerank time per query (20 candidates): {sum(timing['rerank']) / len(timing['rerank']):.2f}s")
    print("\nOverall (150 queries)\n", overall.to_string())
    print("\nans@3 by language\n", per_q.pivot_table(index="system", columns="lang", values="ans@3").round(2).to_string())
    print("\nhit@3 by language\n", per_q.pivot_table(index="system", columns="lang", values="hit@3").round(2).to_string())
    print("\nans@3 by domain\n", per_q.pivot_table(index="system", columns="domain", values="ans@3").round(2).to_string())
    print("\nqueries whose top chunk is for another nation:\n",
          per_q[per_q.top1_nation != "england"].groupby("system").size().to_string())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile eval/translation_review.csv
qid,lang,question,nllb_english,original_english,issue,note
H01,ro,Poate proprietarul să mă evacueze fără să-mi dea un motiv?,Can the owner evacuate me without giving me a reason?,Can my landlord evict me without giving a reason?,key term lost,"""evict"" -> ""evacuate"""
H07,ur,کیا مالک مکان پر کرائے کے گھر کو اچھی حالت میں رکھنا لازم ہے؟,Is it necessary to keep a rented house in good condition?,Does my landlord have to keep my rented home in good repair?,key term lost,landlord dropped
H09,ur,اگر میں اکیلا رہتا ہوں تو کیا مجھے کونسل ٹیکس میں رعایت ملتی ہے؟,"If I live alone, do I get a tax discount?",Do I get a council tax discount if I live alone?,key term lost,"""council tax"" -> ""tax"""
H10,ro,Ce pot face dacă consiliul local sau asociația de locuințe vrea să mă evacueze?,What can I do if the local council or housing association wants to evacuate me?,What can I do if my council or housing association wants to evict me?,key term lost,"""evict"" -> ""evacuate"""
M09,ur,کیا کم آمدنی کی صورت میں مجھے علاج کے اخراجات میں مدد مل سکتی ہے؟,"If I have a low income, can I help with medical expenses?",Can I get help with health costs if I have a low income?,key term lost,"""can I get help"" -> ""can I help"""
M10,ar,كم تبلغ رسوم الرعاية الصحية للمهاجرين بالنسبة للطلاب؟,How much is immigrant health care for students?,How much is the immigration health surcharge for students?,key term lost,"""surcharge"" -> ""health care"""
M10,pl,Ile wynosi dopłata zdrowotna dla imigrantów w przypadku studentów?,How much is immigrant health care for students?,How much is the immigration health surcharge for students?,key term lost,"""surcharge"" -> ""health care"""
M10,ro,Cât costă taxa de sănătate pentru imigranți în cazul studenților?,How much does immigrant health care cost for students?,How much is the immigration health surcharge for students?,key term lost,"""surcharge"" -> ""health care"""
H01,ar,هل يمكن للمالك أن يطردني دون إبداء سبب؟,Can the owner fire me without giving a reason?,Can my landlord evict me without giving a reason?,meaning changed,"""evict"" -> ""fire"" (يطرد means both)"
H03,ar,أخذ المالك مني مبلغ تأمين، هل يجب أن يكون هذا التأمين محميًا؟,The owner took insurance money from me. Does that insurance have to be protected?,My landlord took a deposit from me. Does it have to be protected?,meaning changed,"""deposit"" -> ""insurance money"""
H03,pl,Wynajmujący pobrał ode mnie kaucję. Czy musi ona być zabezpieczona?,The landlord took my bail. Does it have to be secured?,My landlord took a deposit from me. Does it have to be protected?,meaning changed,"""deposit"" (kaucja) -> ""bail"""
H03,ro,Proprietarul mi-a luat o garanție. Trebuie aceasta să fie protejată?,The landlord got me a warranty. Does this have to be protected?,My landlord took a deposit from me. Does it have to be protected?,meaning changed,"""deposit"" (garanție) -> ""warranty"""
H06,ur,میرا گھر چھننے والا ہے، کیا کونسل میری مدد کر سکتی ہے؟,"My house is on the verge of collapse, can the council help me?",I am about to lose my home. Can the council help me?,meaning changed,"""about to lose my home"" -> ""house on the verge of collapse"""
H07,pl,Czy wynajmujący musi utrzymywać wynajmowane mieszkanie w dobrym stanie technicznym?,Does the tenant have to keep the rented apartment in good technical condition?,Does my landlord have to keep my rented home in good repair?,meaning changed,obligation moved from landlord to tenant
H08,ur,گھر آنے سے پہلے مالک مکان کو مجھے کتنا پہلے اطلاع دینی چاہیے؟,How much should I inform the landlord before I return home?,How much notice must my landlord give before visiting the property?,meaning changed,roles reversed: I inform the landlord before I return home
H10,ar,ماذا يمكنني أن أفعل إذا أراد المجلس المحلي أو جمعية الإسكان طردي؟,What can I do if the local council or the housing association wants me fired?,What can I do if my council or housing association wants to evict me?,meaning changed,"""evict"" -> ""fired"""
S02,pl,Do kiedy trzeba złożyć wniosek o miejsce w szkole średniej?,When do you have to apply for a high school job?,What is the last date to apply for a secondary school place?,meaning changed,"""school place"" -> ""high school job"""
S08,pl,W jakim wieku moje dziecko może skończyć szkołę?,How old can my child go to school?,At what age can my child leave school?,meaning changed,"""leave school"" -> ""go to school"""
S09,ur,خصوصی تعلیمی ضروریات والے میرے بچے کو کیا مدد مل سکتی ہے؟,What can I do to help my child with special educational needs?,What support can my child get for special educational needs?,meaning changed,"""what support can my child get"" -> ""what can I do to help my child"""


In [ ]:
!{sys.executable} src/ingest.py

In [ ]:
!{sys.executable} src/chunk.py

In [ ]:
!{sys.executable} eval/build_queries.py

In [ ]:
!{sys.executable} eval/evaluate_week5.py 2>&1 | grep -v -i -E 'warn|unauthenticated|Loading weights'

In [ ]:
import pathlib
import pandas as pd
ROOT = pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
pd.set_option('display.max_colwidth', 90)

## 1. Language identification

`src/language.py` uses lingua restricted to the five project languages. Arabic and Urdu share a script, so restricting the set matters most for those two.

In [ ]:
from language import detect
for q in ['How do I register with a GP surgery?', 'Jak zarejestrować się u lekarza rodzinnego?',
          'Cum mă înregistrez la un medic de familie?', 'كيف أسجل لدى طبيب عام؟', 'میں جی پی کے پاس رجسٹریشن کیسے کرواؤں؟']:
    print(detect(q), '|', q)
pd.read_csv(ROOT / 'results/week5_language_id.csv')

## 2. Translation to English (NLLB-200)

`src/translate.py` splits text into sentences first: NLLB is trained on single sentences and silently dropped the second sentence of two-sentence questions before this fix.

In [ ]:
tr = pd.read_csv(ROOT / 'eval/translations_nllb.csv')
tr[tr.qid.isin(['M01', 'H06'])][['qid', 'lang', 'question', 'nllb_english']]

Manual review of all 120 translations (by the author, not a native speaker): **11 change the meaning** and **8 lose a key term**. Most errors are in housing vocabulary (deposit, evict).

In [ ]:
review = pd.read_csv(ROOT / 'eval/translation_review.csv')
print(review.issue.value_counts().to_string())
review[review.issue == 'meaning changed'][['qid', 'lang', 'nllb_english', 'note']]

## 3. England filter

`src/chunk.py` tags each section by the UK nation named in its heading. Six chunks are for Scotland, Wales or Northern Ireland; search keeps England / UK-wide chunks only.

In [ ]:
chunks = pd.read_csv(ROOT / 'data/processed/chunks.csv')
print(chunks.nation.value_counts().to_string())
chunks[chunks.nation != 'england'][['title', 'heading', 'nation']]

## 4. Full search pipeline

`src/pipeline.py`: detect language > translate > search with both the original and the English question > fuse the two lists (reciprocal rank fusion) > keep England chunks > rerank the top 20 with bge-reranker-v2-m3 > top 3 passages.

In [ ]:
from pipeline import SearchPipeline
pipe = SearchPipeline()

def show(q):
    r = pipe.search(q)
    print(f"Q [{r['lang']}] {q}\n   NLLB English: {r['english_query']}")
    for x in r['results']:
        print(f"   - {x['title']} > {x['heading'][:60]}  [{x['nation']}]\n     {x['text'][:140]}...")
    print()

show('At what age can my child leave school?')             # week 4 returned the Scotland rule first
show('هل يمكن للمالك أن يطردني دون إبداء سبب؟')              # Arabic: NLLB says 'fire me', search still finds eviction rules
show('میرا گھر چھننے والا ہے، کیا کونسل میری مدد کر سکتی ہے؟')  # Urdu: NLLB says 'collapse', search still finds homelessness help

## 5. Experiment: six set-ups on the 150-query test set

Same metrics as week 4. **ans@3** = a top-3 chunk contains the verbatim answer phrase. Regenerate with `python eval/evaluate_week5.py`.

In [ ]:
res = pd.read_csv(ROOT / 'results/week5_by_lang.csv')
res[res.lang == 'ALL'].drop(columns='lang').set_index('system')

In [ ]:
per_q = pd.read_csv(ROOT / 'results/week5_per_query.csv')
langs = ['en', 'pl', 'ro', 'ar', 'ur']
per_q.pivot_table(index='system', columns='lang', values='ans@3').round(2)[langs]

In [ ]:
import matplotlib.pyplot as plt
show_sys = ['A direct', 'C translated+filter', 'E direct+filter+rerank', 'F fusion+filter+rerank']
t = per_q[per_q.system.isin(show_sys)].pivot_table(index='lang', columns='system', values='ans@3').loc[langs, show_sys]
ax = t.plot.bar(figsize=(8, 3.5), rot=0, width=0.8)
ax.set_ylabel('ans@3'); ax.set_ylim(0, 1); ax.set_title('Answer-bearing chunk in top 3, by query language')
ax.legend(frameon=False, ncol=2, fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.12))
plt.tight_layout(); plt.show()

In [ ]:
print('queries whose top chunk is for another UK nation:')
per_q[per_q.top1_nation != 'england'].groupby('system').size()

## Findings

* Language ID was correct for all 150 test questions (clean, full sentences; real users may mix languages).
* Translating first helps Urdu most (ans@3 0.47 to 0.73) even though 19 of 120 translations have errors.
* Reranking gives the largest single gain (0.68 to 0.87 with the filter).
* The chosen set-up, **F fusion+filter+rerank**, reaches hit@3 1.00 and ans@3 0.89, with all five languages at 0.87 or above.
* No top result comes from another UK nation's rules once the filter is on.

## Next (week 6)
Give the top passages to an LLM, answer only from them with numbered citations, refuse when nothing relevant is found, and return the answer in the user's language.